<a href="https://colab.research.google.com/github/Al-Amin95/PromptInjectionDetectionSystem/blob/data-analysis/notebooks/1-data-analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1 Data Analysis: Prompt Injection Detection Dataset

## Part-1 A — Notebook Setup and Reproducibility

In [ ]:
# 1. Mount Google Drive

from google.colab import drive
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")
MY_DRIVE = DRIVE_ROOT / "MyDrive"

if MY_DRIVE.exists():
    print("Google Drive is already mounted.")
else:
    drive.mount("/content/drive")
    print("Google Drive mounted successfully.")

print("MyDrive path:", MY_DRIVE)
print("MyDrive exists:", MY_DRIVE.exists())

In [ ]:
# 2. Auto-detect project folder path

from pathlib import Path

MY_DRIVE = Path("/content/drive/MyDrive")

print("MyDrive exists:", MY_DRIVE.exists())

# 1. Find USW dissertation folder automatically
usw_candidates = []

for item in MY_DRIVE.iterdir():
    if item.is_dir():
        normalized_name = item.name.lower().replace("_", " ").replace("-", " ")
        if "usw" in normalized_name and "dissertation" in normalized_name:
            usw_candidates.append(item)

print("\nUSW candidate folders:")
for idx, folder in enumerate(usw_candidates):
    print(idx, "->", repr(folder.name), "| full path:", folder)

if len(usw_candidates) == 0:
    raise FileNotFoundError("No USW Dissertation folder found inside MyDrive.")

USW_DIR = usw_candidates[0]

# 2. Find project folder automatically
project_candidates = []

for item in USW_DIR.iterdir():
    if item.is_dir():
        normalized_name = item.name.lower().replace("_", " ").replace("-", " ")
        if "prompt" in normalized_name and "injection" in normalized_name:
            project_candidates.append(item)

print("\nProject candidate folders:")
for idx, folder in enumerate(project_candidates):
    print(idx, "->", repr(folder.name), "| full path:", folder)

if len(project_candidates) == 0:
    raise FileNotFoundError("No Prompt Injection project folder found inside the USW folder.")

PROJECT_ROOT = project_candidates[0]

# 3. Find Notebooks folder automatically, case-insensitive
notebook_candidates = []

for item in PROJECT_ROOT.iterdir():
    if item.is_dir() and item.name.lower() == "notebooks":
        notebook_candidates.append(item)

print("\nNotebook candidate folders:")
for idx, folder in enumerate(notebook_candidates):
    print(idx, "->", repr(folder.name), "| full path:", folder)

if len(notebook_candidates) == 0:
    raise FileNotFoundError("No Notebooks folder found inside the project folder.")

NOTEBOOKS_DIR = notebook_candidates[0]

print("\nFINAL PATHS")
print("USW_DIR:", USW_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("NOTEBOOKS_DIR:", NOTEBOOKS_DIR)

print("\nCHECKS")
print("USW_DIR exists:", USW_DIR.exists())
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())
print("NOTEBOOKS_DIR exists:", NOTEBOOKS_DIR.exists())

In [ ]:
# 3-GitHub connection Check

USE_GITHUB = True

GITHUB_USERNAME = "AI-Amin95"
GITHUB_REPOSITORY = "PromptInjectionDetectionSystem"
GITHUB_BRANCH = "data-analysis"
GITHUB_NOTEBOOK_PATH = "notebooks/1-data-analysis.ipynb"

GITHUB_REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPOSITORY}"

print("GitHub connection confirmed.")
print("Repository:", GITHUB_REPO_URL)
print("Branch:", GITHUB_BRANCH)
print("Notebook path:", GITHUB_NOTEBOOK_PATH)
print("Save method: Use Colab 'Save in GitHub to keep changes'.")

In [ ]:
# 4. Import libraries for data analysis

import os
import sys
import re
import json
import random
import platform
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

In [ ]:
# 5. Set random seed

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print(f"Random seed set to: {SEED}")

In [ ]:
# 7. Record environment information

environment_info = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "matplotlib_version": matplotlib.__version__,
    "notebook_run_datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "random_seed": SEED
}

for key, value in environment_info.items():
    print(f"{key}: {value}")

In [ ]:
# 6. Define project paths


DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
DATA_ANALYSIS_OUTPUT_DIR = OUTPUTS_DIR / "data_analysis"

TABLES_DIR = DATA_ANALYSIS_OUTPUT_DIR / "tables"
FIGURES_DIR = DATA_ANALYSIS_OUTPUT_DIR / "figures"
REPORTS_DIR = DATA_ANALYSIS_OUTPUT_DIR / "reports"
LOGS_DIR = DATA_ANALYSIS_OUTPUT_DIR / "logs"

MODELS_DIR = PROJECT_ROOT / "models"
APP_DIR = PROJECT_ROOT / "webapp"

print("Project paths defined successfully.")
print("Project root:", PROJECT_ROOT)
print("Notebooks directory:", NOTEBOOKS_DIR)
print("Raw data directory:", RAW_DATA_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)
print("Tables directory:", TABLES_DIR)
print("Figures directory:", FIGURES_DIR)
print("Reports directory:", REPORTS_DIR)
print("Logs directory:", LOGS_DIR)
print("Models directory:", MODELS_DIR)
print("App directory:", APP_DIR)

In [ ]:
# 7 Create required folders

project_dirs = [
    PROJECT_ROOT,
    NOTEBOOKS_DIR,
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    OUTPUTS_DIR,
    DATA_ANALYSIS_OUTPUT_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    REPORTS_DIR,
    LOGS_DIR,
    MODELS_DIR,
    APP_DIR
]

for directory in project_dirs:
    directory.mkdir(parents=True, exist_ok=True)

print("Required folders created/confirmed successfully.")

In [ ]:
# 8 Check folder structure

folder_check = []

for directory in project_dirs:
    folder_check.append({
        "folder_name": directory.name,
        "folder_path": str(directory),
        "exists": directory.exists()
    })

folder_check_df = pd.DataFrame(folder_check)

folder_check_df

In [ ]:
# 9 Save folder structure check

folder_check_path = LOGS_DIR / "folder_structure_check.csv"

folder_check_df.to_csv(folder_check_path, index=False)

print("Folder structure check saved to:")
print(folder_check_path)
print("File exists:", folder_check_path.exists())

In [ ]:
# 10. Save analysis_config.json

analysis_config = {
    "project_title": "Prompt Injection Detection Using Fine-Tuned Transformer Models: A Glass-Box Approach with Explainable AI",
    "notebook_name": "1-data-analysis.ipynb",
    "analysis_stage": "Part 1 - Data Analysis",
    "dataset_name": "deepset/prompt-injections",
    "task": "Binary prompt injection classification",
    "label_mapping": {
        "0": "SAFE",
        "1": "INJECTION"
    },
    "random_seed": SEED,
    "raw_data_policy": "Read-only in Part 1. No permanent modification in this notebook.",
    "github_repository": GITHUB_REPO_URL,
    "github_branch": GITHUB_BRANCH,
    "github_notebook_path": GITHUB_NOTEBOOK_PATH,
    "project_root": str(PROJECT_ROOT),
    "notebooks_dir": str(NOTEBOOKS_DIR),
    "raw_data_dir": str(RAW_DATA_DIR),
    "processed_data_dir": str(PROCESSED_DATA_DIR),
    "tables_dir": str(TABLES_DIR),
    "figures_dir": str(FIGURES_DIR),
    "reports_dir": str(REPORTS_DIR),
    "logs_dir": str(LOGS_DIR),
    "models_dir": str(MODELS_DIR),
    "app_dir": str(APP_DIR),
    "folder_structure_check_file": str(folder_check_path),
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "python_version": sys.version,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "matplotlib_version": matplotlib.__version__
}

config_path = LOGS_DIR / "analysis_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(analysis_config, f, indent=4)

print("Analysis configuration saved to:")
print(config_path)
print("File exists:", config_path.exists())